In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2. 修复二进制依赖：安装系统级 ffmpeg (Colab通常自带，但保险起见更新/确认一下)
!apt-get update && apt-get install ffmpeg -y

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,308 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,083 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,332 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://r2u.stat.illinois

In [3]:
# 3. 安装并配置 uv
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 105.5 MB/s eta 0:00:00


In [6]:
%cd /content/drive/MyDrive/ma-zhaoyi-shi/

/content/drive/MyDrive/ma-zhaoyi-shi


In [7]:
!ls

artifacts			 kpi		 supervised_training.py
colab_init.ipynb		 kpi.py		 train_example.py
demo_expriment.py		 __pycache__	 train.log
demo_lstm_experiment.py		 pyproject.toml  uv.lock
demo_lstm_predict_experiment.py  README.md
export_text_label_dataset.py	 results


In [8]:
# 4. 核心：同步环境
# 如果你需要用到 cupy (GPU加速)，请确保 Colab 开启了 GPU，并运行下面这行：
!UV_SYSTEM_PYTHON=1 uv sync --project . --extra gpu

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 102 packages in 3ms
Prepared 99 packages in 45.93s
░░░░░░░░░░░░░░░░░░░░ [0/99] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 99 packages in 7m 24s
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.14.1
 + audioread==3.1.0
 + blis==1.3.3
 + catalogue==2.0.10
 + certifi==2026.6.17
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + click==8.4.2
 + cloudpathlib==0.24.0
 + confection==1.3.3
 + cupy-cuda12x==13.6.0
 + cymem==2.0.13
 + decorator==5.3.1
 + fastrlock==0.8.3
 + ffmpeg-python==0.2.0
 + filelock==3.29.5
 + fsspec==2026.6.0
 + future==1.

In [9]:
import torch
print("CUDA 是否可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("当前 GPU 设备:", torch.cuda.get_device_name(0))

CUDA 是否可用: True
当前 GPU 设备: Tesla T4


In [11]:
from pathlib import Path

path = Path("/content/drive/MyDrive/mitfld_processed.pt")
print(path.exists(), path.stat().st_size if path.exists() else None)

True 38312260


In [16]:
!uv run python supervised_training.py \
  --processed_dataset_path "/content/drive/MyDrive/ma-zhaoyi-shi/processed_dataset/mitfld_processed.pt" \
  --epochs 20 \
  --batch_size 4 \
  --lr 1e-4 \
  --output_dir "/content/drive/MyDrive/stb_runs" \
  --log_level DID

/content/drive/MyDrive/ma-zhaoyi-shi/kpi/models/_twfinch.py:14: UserWarning: pynndescent not installed: No module named 'pynndescent'
  warnings.warn('pynndescent not installed: {}'.format(e))
2026-07-08 09:46:35 | INFO | root | Logging initialized. Log file: /content/drive/MyDrive/ma-zhaoyi-shi/kpi/utils/train.log
2026-07-08 09:46:35 | INFO | kpi.utils.stb_supervised.config_utils | Split ratios validated successfully
2026-07-08 09:46:35 | INFO | kpi.utils.stb_supervised.config_utils | Resolved device=auto to cuda
2026-07-08 09:46:35 | INFO | __main__ | Using device: cuda
2026-07-08 09:46:35 | INFO | __main__ | Loading processed dataset from /content/drive/MyDrive/ma-zhaoyi-shi/processed_dataset/mitfld_processed.pt
2026-07-08 09:46:35 | INFO | kpi.models.STB.sentence_encoder | Initializing SentenceEncoder(model_name=all-MiniLM-L6-v2, embedding_dim=384, max_length=256, normalize_embeddings=True, cache_folder=None)
2026-07-08 09:46:35 | INFO | sentence_transformers.SentenceTransformer | 